<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/HVG_creation_250326.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this file, we
create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

In [5]:
#!pip install ts2vg mne

from pathlib import Path
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import save_npz
from ts2vg import HorizontalVG
import mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.9 MB/s eta 0:00:00


In [6]:
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length

def process_edf_with_labels(edf_file, seizure_file=None):
    edf_file = Path(edf_file)
    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.)
    raw.set_eeg_reference('average')

    sfreq = raw.info["sfreq"]
    n_samples = len(raw.times)
    time_marks = np.arange(n_samples) / sfreq
    electrode_data = raw.get_data()
    electrode_names = raw.ch_names

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = "Time (s)"

    if seizure_file is not None:
        seizure_start, seizure_length = get_seizure_period(Path(seizure_file))
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
    else:
        df["label"] = "interictal"

    return df

In [10]:
%cd '/Users/antoniaspoerk/Desktop/Digital Neuroscience /Social Media Analytics/epilepsy_pediatrics_EEG/data/raw'
RAW_DIR = Path(".")  # adjust to your actual path

df = process_edf_with_labels(
    edf_file=RAW_DIR / "chb01_03.edf",
    seizure_file=RAW_DIR / "chb01_03.edf.seizures"
)

print(df.shape)
print(df["label"].value_counts())

/Users/antoniaspoerk/Desktop/Digital Neuroscience /Social Media Analytics/epilepsy_pediatrics_EEG/data/raw


/var/folders/9m/vzs4mddj1tq0m3tmypn79plc0000gn/T/ipykernel_51248/1677169844.py:10: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
(921600, 24)
label
interictal    911359
ictal          10241
Name: count, dtype: int64


In [11]:
# extract 2990s–3020s window
def extract_window_by_time(df, start_time_s, window_size_s=30):
    dt = float(df.index[1] - df.index[0])
    sfreq = int(round(1 / dt))
    window_size_samples = int(round(window_size_s * sfreq))

    index_values = np.asarray(df.index, dtype=float)
    start_idx = int(np.argmin(np.abs(index_values - start_time_s)))
    end_idx = start_idx + window_size_samples

    if end_idx > len(df):
        raise ValueError("Window exceeds file length.")

    return df.iloc[start_idx:end_idx].copy(), start_idx

window_df, start_idx = extract_window_by_time(df, start_time_s=2990.0, window_size_s=30)

print(f"Window: {window_df.index[0]:.1f}s → {window_df.index[-1]:.1f}s")
print(f"Samples: {len(window_df)}")
print(f"Label counts:\n{window_df['label'].value_counts()}")

Window: 2990.0s → 3020.0s
Samples: 7680
Label counts:
label
ictal         6144
interictal    1536
Name: count, dtype: int64
